In [1]:
!pip install bayesian-optimization
!pip install scikit-optimize
!pip install hyperopt
!pip install ConfigSpace
!pip install smac
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
from bayes_opt import BayesianOptimization
from bayes_opt.acquisition import ExpectedImprovement
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from skopt import forest_minimize
from skopt.space import Real
from ConfigSpace import ConfigurationSpace, Float
from smac import HyperparameterOptimizationFacade, Scenario

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.6/117.6 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.9/172.9 kB 5.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.0 MB/s eta 0:00:00
  Created wheel for smac: filename=smac-2.4.1-py3-none-any.whl size=236299 sha256=b47e94b703d649a233f6647d77b2b5e92731c81667a595c708045f53fa894de8
  Stored in directory: /ro

## Patient response function and Bayesian optimzer algorithm functions.

In [2]:
import pandas as pd
import numpy as np
from bayes_opt import BayesianOptimization
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from ConfigSpace import ConfigurationSpace, Float
from smac import HyperparameterOptimizationFacade, Scenario

# Here are many different families of patients.

baseline_family = {
    'name': 'quick_rise_slow_fall',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

symmetric_family = {
    'name': 'symmetric',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'symmetric',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

slow_rise_quick_fall_family = {
    'name': 'slow_rise_quick_fall',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'slow_rise_quick_fall',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

three_optima_family = {
    'name': '3optima',
    'num_optima': 3,
    'spread_scale': 1.0,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

broad_peaks_family = {
    'name': 'broad_optima',
    'num_optima': 2,
    'spread_scale': 2.0,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

sharp_peaks_family = {
    'name': 'sharp_optima',
    'num_optima': 2,
    'spread_scale': 0.4,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

heteroscedastic_noise_family = {
    'name': 'heteroscedastic_noise',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'heteroscedastic',
    'unique_parameter_interaction': 'none'
}

heavy_tailed_noise_family = {
    'name': 'heavy_tailed_noise',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'heavy_tailed',
    'unique_parameter_interaction': 'none'
}

unique_parameter_interaction_family = {
    'name': 'unique_parameter_interaction',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'normal',
    'unique_parameter_interaction': 'pulsewidth-amplitude'
}


def asymmetric_hill_peak(
    value,
    optimum,
    left_scale,
    right_scale,
    hill_n=4.0,
):
    """Return a Hill-like peak anchored at an explicit optimum."""
    if left_scale <= 0 or right_scale <= 0 or hill_n <= 0:
        raise ValueError("Scales and hill_n must be positive.")

    value = np.asarray(value, dtype=float)
    scale = np.where(value < optimum, left_scale, right_scale)
    distance = np.abs(value - optimum) / scale

    return 1.0 / (1.0 + distance**hill_n)


def generate_patient_profile(patient_seed: int, family: dict):
    """Draws one unique synthetic patient's response landscape for a given surface family, independent of any optimizer's own RNG."""
    rng = np.random.default_rng(patient_seed)

    base_optima = [(10.0 + rng.uniform(-2.0, 2.0), 500.0 + rng.uniform(-25.0, 25.0)), 
                   (25.0 + rng.uniform(-2.0, 2.0), 100.0 + rng.uniform(-25.0, 25.0))] #may need to change this later on.
    num_optima = family['num_optima']
    optima = []
    for numOfOptima in range(num_optima):
        if numOfOptima < len(base_optima):
            opt_f, opt_w = base_optima[numOfOptima]
        else:
            opt_f = rng.uniform(5.0, 50.0)
            opt_w = rng.uniform(150.0, 450.0)
        optima.append((opt_f, opt_w))

    spread_scale = family['spread_scale']

    if family['function_shape'] == 'quick_rise_slow_fall':
        return {
            'patient_seed': patient_seed,
            'surface_family': family['name'],
            'ideal_amp_pup': rng.normal(4.4, 0.15),
            'ideal_amp_hrv': rng.normal(4.0, 0.15),
            # Asymmetric spreads: faster rise toward threshold, slower decline above it
            'amp_rise_pup': rng.uniform(0.25, 0.35) * spread_scale,
            'amp_fall_pup': rng.uniform(0.55, 0.75) * spread_scale,
            'amp_rise_hrv': rng.uniform(0.20, 0.30) * spread_scale,
            'amp_fall_hrv': rng.uniform(0.45, 0.65) * spread_scale,

            'freq_rise': rng.uniform(1.0, 2.0) * spread_scale,
            'freq_fall': rng.uniform(5.0, 6.0) * spread_scale,
            'pulse_width_rise': rng.uniform(20.0, 50.0) * spread_scale,
            'pulse_width_fall': rng.uniform(125.0, 175.0) * spread_scale,

            'optima': optima,
        }
    elif family['function_shape'] == 'symmetric':
        return {
            'patient_seed': patient_seed,
            'surface_family': family['name'],
            'ideal_amp_pup': rng.normal(4.4, 0.15),
            'ideal_amp_hrv': rng.normal(4.0, 0.15),
            # Symmetric spreads: same rise and fall
            'amp_rise_pup': 0.25,
            'amp_fall_pup': 0.25,
            'amp_rise_hrv': 0.25,
            'amp_fall_hrv': 0.25,
            'freq_rise': 1.5,
            'freq_fall': 1.5,
            'pulse_width_rise': 35.0,
            'pulse_width_fall': 150.0,

            'optima': optima,
        }
    elif family['function_shape'] == 'slow_rise_quick_fall':
        return {
            'patient_seed': patient_seed,
            'surface_family': family['name'],
            'ideal_amp_pup': rng.normal(4.4, 0.15),
            'ideal_amp_hrv': rng.normal(4.0, 0.15),
            # Asymmetric spreads: slower rise toward threshold, faster decline above it
            'amp_rise_pup': rng.uniform(0.55, 0.75) * spread_scale,
            'amp_fall_pup': rng.uniform(0.25, 0.35) * spread_scale,
            'amp_rise_hrv': rng.uniform(0.45, 0.65) * spread_scale,
            'amp_fall_hrv': rng.uniform(0.20, 0.30) * spread_scale,

            'freq_rise': rng.uniform(5.0, 6.0) * spread_scale,
            'freq_fall': rng.uniform(1.0, 2.0) * spread_scale,
            'pulse_width_rise': rng.uniform(125.0, 175.0) * spread_scale,
            'pulse_width_fall': rng.uniform(20.0, 50.0) * spread_scale,

            'optima': optima,
        }


def run_optimization_comparison(num_simulation_trials: int, amount_of_patients: int, patient_and_machine_startSeed : int, patient_type: dict):

    search_boundaries = {
        'amplitude': (0.5, 5.0),       # mA
        'frequency': (1.0, 50.0),      # Hz
        'pulse_width': (50.0, 600.0),  # microseconds
    }
    
    all_patients_results = []  # accumulates each patient's comparison_data across the loop
    all_convergence_results = []

    for i in range(amount_of_patients):

        patient_profile = generate_patient_profile(patient_and_machine_startSeed, patient_type)
        ideal_amp_pup = patient_profile['ideal_amp_pup']
        amp_rise_pup = patient_profile['amp_rise_pup']
        amp_fall_pup = patient_profile['amp_fall_pup']
        ideal_amp_hrv = patient_profile['ideal_amp_hrv']
        amp_rise_hrv = patient_profile['amp_rise_hrv']
        amp_fall_hrv = patient_profile['amp_fall_hrv']
        freq_rise = patient_profile['freq_rise']
        freq_fall = patient_profile['freq_fall']
        pulse_width_rise = patient_profile['pulse_width_rise']
        pulse_width_fall = patient_profile['pulse_width_fall']
        optima = patient_profile['optima']
  

        def realistic_patient_curve(amplitude, frequency, pulse_width, AddNoise: bool, patient_and_machine_startSeed: int):

            rng = np.random.default_rng(patient_and_machine_startSeed)
            
            # Pupillometry Calculation (anchored asymmetric Hill peaks)
            amp_comp_pup = asymmetric_hill_peak(
                amplitude,
                ideal_amp_pup,
                amp_rise_pup,
                amp_fall_pup,
            )
            amp_comp_hrv = asymmetric_hill_peak(
                amplitude,
                ideal_amp_hrv,
                amp_rise_hrv,
                amp_fall_hrv,
            )

            pup_score = 100 * amp_comp_pup 
            hrv_score = 100 * amp_comp_hrv 
            w_pup, w_hrv = 0.5, 0.5
            amplitude_score = w_pup * pup_score + w_hrv * hrv_score

            # We calculate response based on proximity to the optimal (Freq, Width) pairs
            def get_interdependence_score(f, w, Amplitude_Scale_Factor = 1.0):
                scores = []
                if Amplitude_Scale_Factor != 1.0:
                    for (opt_f, opt_w) in optima:
                        freq_comp = asymmetric_hill_peak(
                            f,
                            opt_f,
                            freq_rise,
                            freq_fall,
                        )
                        width_comp = asymmetric_hill_peak(
                            w,
                            opt_w,
                            pulse_width_rise,
                            pulse_width_fall,
                        )
                        dist = ((freq_comp) * (width_comp)) * Amplitude_Scale_Factor
                        scores.append(dist)
                else:
                    for (opt_f, opt_w) in optima:
                        freq_comp = asymmetric_hill_peak(
                            f,
                            opt_f,
                            freq_rise,
                            freq_fall,
                        )
                        width_comp = asymmetric_hill_peak(
                            w,
                            opt_w,
                            pulse_width_rise,
                            pulse_width_fall,
                        )
                        dist = ((freq_comp) *  (width_comp))
                        scores.append(dist)

                return max(scores)

            if patient_type.get('unique_parameter_interaction') == 'pulsewidth-amplitude':
                inter_score = get_interdependence_score(frequency, pulse_width, amplitude_score/100)
            else:
                inter_score = get_interdependence_score(frequency, pulse_width)

            total_score = inter_score * amplitude_score

            #Now calculate best possible score:
            amplitude_grid = np.linspace(0.5, 5.0, 10_000)
            amp_comp_pup_grid = asymmetric_hill_peak(
                amplitude_grid,
                ideal_amp_pup,
                amp_rise_pup,
                amp_fall_pup,
            )

            amp_comp_hrv_grid = asymmetric_hill_peak(
                amplitude_grid,
                ideal_amp_hrv,
                amp_rise_hrv,
                amp_fall_hrv,
            )

            amplitude_scores = (
                0.5 * 100 * amp_comp_pup_grid + 0.5 * 100 * amp_comp_hrv_grid)

            best_amplitude_score = np.max(amplitude_scores)
            best_amplitude = amplitude_grid[np.argmax(amplitude_scores)]

            #1 assumes the best combo of frequency and pulse width, just for show.
            best_possible_score = best_amplitude_score * 1


            if not AddNoise:
                return total_score
             
            noise_type = patient_type.get('noise', 'normal')
            if noise_type == 'normal':
                noise = rng.normal(0, 2.0)  # e.g., standard deviation of 2.0
            elif noise_type == 'heteroscedastic':
                    # Noise scales with signal magnitude
                noise = rng.normal(0, 0.05 * amplitude_score)
            elif noise_type == 'heavy_tailed':
                    # Student's t-distribution
                noise = rng.standard_t(df=3) * 2.0
            else:
                noise = 0.0

            total_score += noise
        
            return total_score
        
        #Find the optimal score and parameters
        amplitude_grid = np.linspace(search_boundaries['amplitude'][0], search_boundaries['amplitude'][1], 10000)
        
        max_possible_score = -np.inf
        true_optimal_params = {}

        for (opt_f, opt_w) in optima:
            # Vectorized evaluation over the amplitude grid at the peak frequency & pulse width
            curve_scores = realistic_patient_curve(amplitude_grid, opt_f, opt_w, False, 0)
            best_idx = np.argmax(curve_scores)
            peak_score = curve_scores[best_idx]
            
            if peak_score > max_possible_score:
                max_possible_score = peak_score
                true_optimal_params = {
                    'amplitude': amplitude_grid[best_idx],
                    'frequency': opt_f,
                    'pulse_width': opt_w
                }

        method_codes = {
            "Bayesian Optimization": 1,
            "TPE Optimization": 2,
            "SMAC Optimization": 3,
            "Random Search": 4,
        }

        def make_evaluation_seed(profile_seed, query_index):
            sequence = np.random.SeedSequence(
                [int(profile_seed), int(query_index)]
            )
            return int(sequence.generate_state(1, dtype=np.uint32)[0])

        profile_seed = patient_and_machine_startSeed
        convergence_rows = []

        query_counts = {
            method: 0
            for method in method_codes
        }

        best_observed = {
            method: -np.inf
            for method in method_codes
        }

        best_noise_free = {
            method: -np.inf
            for method in method_codes
        }

        incumbent_noise_free = {
            method: np.nan
            for method in method_codes
        }

        def evaluate_and_record(method, params, evaluation_seed):
            query_index = query_counts[method] + 1
            query_counts[method] = query_index

            observed_score = realistic_patient_curve(
                params["amplitude"],
                params["frequency"],
                params["pulse_width"],
                True,
                evaluation_seed,
            )

            noise_free_score = realistic_patient_curve(
                params["amplitude"],
                params["frequency"],
                params["pulse_width"],
                False,
                evaluation_seed,
            )

            is_new_observed_best = observed_score > best_observed[method]

            best_observed[method] = max(
                best_observed[method],
                observed_score,
            )

            best_noise_free[method] = max(
                best_noise_free[method],
                noise_free_score,
            )

            if is_new_observed_best:
                incumbent_noise_free[method] = noise_free_score

            convergence_rows.append({
                "Profile ID": (
                    f"{patient_profile['surface_family']}::{profile_seed}"
                ),
                "Surface Family": patient_profile["surface_family"],
                "Profile Seed": profile_seed,
                "Iteration Budget": num_simulation_trials,
                "Method": method,
                "Query": query_index,
                "Evaluation Seed": evaluation_seed,
                "Amplitude": params["amplitude"],
                "Frequency": params["frequency"],
                "Pulse Width": params["pulse_width"],
                "Observed Score": observed_score,
                "Noise-Free Score": noise_free_score,
                "Cumulative Best Observed Score": best_observed[method],
                "Best Noise-Free Score So Far": best_noise_free[method],
                "Current Incumbent Noise-Free Score": incumbent_noise_free[method],
                "Normalized Regret So Far (%)": (
                    100
                    * (
                        max_possible_score
                        - best_noise_free[method]
                    )
                    / max_possible_score
                ),
            })

            return observed_score

        def objective_bo(amplitude, frequency, pulse_width):
            query_index = (
                query_counts["Bayesian Optimization"] + 1
            )
            evaluation_seed = make_evaluation_seed(
                profile_seed,
                query_index,
            )

            return evaluate_and_record(
                "Bayesian Optimization",
                {
                    "amplitude": amplitude,
                    "frequency": frequency,
                    "pulse_width": pulse_width,
                },
                evaluation_seed,
            )

        optimizer_multi = BayesianOptimization(
            f=objective_bo,
            pbounds=search_boundaries,
            acquisition_function=ExpectedImprovement(xi=0.01),
            random_state=patient_and_machine_startSeed,
            verbose=0
        )
        print(f"\n--- Starting Bayesian Optimization ({num_simulation_trials} iterations) ---")
        bo_start_time = time.perf_counter()
        optimizer_multi.maximize(init_points=max(1, num_simulation_trials // 10), n_iter=num_simulation_trials - max(1, num_simulation_trials // 10))

        bo_elapsed_seconds = time.perf_counter() - bo_start_time
        print(f"Bayesian Optimization completed in {bo_elapsed_seconds:.3f} seconds.")

        # --- TPE Optimization ---
        tpe_start_time = time.perf_counter()
        def objective_tpe(params):
            query_index = query_counts["TPE Optimization"] + 1
            evaluation_seed = make_evaluation_seed(
                profile_seed,
                query_index,
            )

            score = evaluate_and_record(
                "TPE Optimization",
                {
                    "amplitude": params["amplitude"],
                    "frequency": params["frequency"],
                    "pulse_width": params["pulse_width"],
                },
                evaluation_seed,
            )

            return {
                "loss": -score,
                "status": STATUS_OK,
            }
        space = {
            'amplitude': hp.uniform('amplitude', search_boundaries['amplitude'][0], search_boundaries['amplitude'][1]),
            'frequency': hp.uniform('frequency', search_boundaries['frequency'][0], search_boundaries['frequency'][1]),
            'pulse_width': hp.uniform('pulse_width', search_boundaries['pulse_width'][0], search_boundaries['pulse_width'][1])
        }
        trials = Trials()
        print(f"\n--- Starting TPE Optimization ({num_simulation_trials} iterations) ---")
        fmin(fn=objective_tpe, space=space, algo=tpe.suggest, max_evals=num_simulation_trials, trials=trials, rstate=np.random.default_rng(patient_and_machine_startSeed))
        tpe_elapsed_seconds = time.perf_counter() - tpe_start_time
        print(f"TPE completed in {tpe_elapsed_seconds:.3f} seconds.")
        
        # --- SMAC Optimization ---
        smac_start_time = time.perf_counter()
        configspace = ConfigurationSpace(seed=patient_and_machine_startSeed)
        configspace.add([
            Float('amplitude', bounds=search_boundaries['amplitude']),
            Float('frequency', bounds=search_boundaries['frequency']),
            Float('pulse_width', bounds=search_boundaries['pulse_width']),
        ])
        
        def objective_smac(config, seed=None):
            query_index = query_counts["SMAC Optimization"] + 1
            evaluation_seed = make_evaluation_seed(
                profile_seed,
                query_index,
            )
            score = evaluate_and_record(
                "SMAC Optimization",
                {
                    "amplitude": float(config["amplitude"]),
                    "frequency": float(config["frequency"]),
                    "pulse_width": float(config["pulse_width"]),
                },
                evaluation_seed,
            )

            return -score
        
        print(f"\n--- Starting SMAC Optimization ({num_simulation_trials} iterations) ---")
        scenario = Scenario(
            configspace,
            deterministic=False,
            n_trials=num_simulation_trials,
            seed=patient_and_machine_startSeed,
        )
        smac = HyperparameterOptimizationFacade(scenario, objective_smac)
        incumbent = smac.optimize()
        smac_score = -smac.runhistory.get_cost(incumbent)

        smac_elapsed_seconds = time.perf_counter() - smac_start_time
        print(f"SMAC completed in {smac_elapsed_seconds:.3f} seconds.")

        
        # --- Random Search ---
        rs_start_time = time.perf_counter()
        random_search_rng = np.random.default_rng(patient_and_machine_startSeed)
        best_random_score = -np.inf
        best_random_params = {}
        random_search_scores = []  # score at each evaluation, in order, for convergence tracking
        print(f"\n--- Starting Random Search Optimization ({num_simulation_trials} iterations) ---")
        for _ in range(num_simulation_trials):
            a, f, w = [
                random_search_rng.uniform(
                    search_boundaries[key][0],
                    search_boundaries[key][1],
                )
                for key in [
                    "amplitude",
                    "frequency",
                    "pulse_width",
                ]
            ]

            query_index = query_counts["Random Search"] + 1
            evaluation_seed = make_evaluation_seed(
                profile_seed,
                query_index,
            )

            current_score = evaluate_and_record(
                "Random Search",
                {
                    "amplitude": a,
                    "frequency": f,
                    "pulse_width": w,
                },
                evaluation_seed,
            )

            random_search_scores.append(current_score)

            if current_score > best_random_score:
                best_random_score = current_score
                best_random_params = {
                    "amplitude": a,
                    "frequency": f,
                    "pulse_width": w,
                }

        rs_elapsed_seconds = time.perf_counter() - rs_start_time
        print(f"Random Search completed in {rs_elapsed_seconds:.3f} seconds.")

        selected_scores_noise_free = [
            realistic_patient_curve(
                optimizer_multi.max['params']['amplitude'],
                optimizer_multi.max['params']['frequency'],
                optimizer_multi.max['params']['pulse_width'],
                False,
                patient_and_machine_startSeed,
            ),
            realistic_patient_curve(
                trials.argmin['amplitude'],
                trials.argmin['frequency'],
                trials.argmin['pulse_width'],
                False,
                patient_and_machine_startSeed,
            ),
            realistic_patient_curve(
                float(incumbent['amplitude']),
                float(incumbent['frequency']),
                float(incumbent['pulse_width']),
                False,
                patient_and_machine_startSeed,
            ),
            realistic_patient_curve(
                best_random_params['amplitude'],
                best_random_params['frequency'],
                best_random_params['pulse_width'],
                False,
                patient_and_machine_startSeed,
            ),
        ]

        simple_regrets = [
            max_possible_score - score
            for score in selected_scores_noise_free
        ]

        normalized_regrets = [
            regret / max_possible_score
            if max_possible_score > 0
            else np.nan
            for regret in simple_regrets
        ]

        comparison_data = {
            'Method': ['Bayesian Optimization', 'TPE Optimization', 'SMAC Optimization', 'Random Search'],
            'Surface Family': [patient_profile['surface_family']] * 4,
            'Seed': [patient_and_machine_startSeed] * 4,
            'Best Possible Score': [max_possible_score] * 4,
            'Best Computed Score': [optimizer_multi.max['target'], -trials.best_trial['result']['loss'], smac_score, best_random_score],
            'Selected Amplitude (mA)': [optimizer_multi.max['params']['amplitude'], trials.argmin['amplitude'], float(incumbent['amplitude']), best_random_params['amplitude']],
            'Selected Frequency (Hz)': [optimizer_multi.max['params']['frequency'], trials.argmin['frequency'], float(incumbent['frequency']), best_random_params['frequency']],
            'Selected Pulse Width (us)': [optimizer_multi.max['params']['pulse_width'], trials.argmin['pulse_width'], float(incumbent['pulse_width']), best_random_params['pulse_width']],
            
            'Selected Score (Noise-Free)': selected_scores_noise_free,
            'Simple Regret': simple_regrets,
            #Gives percentage regret just in case
            'Normalized Regret (%)': [
                                100 * regret
                                for regret in normalized_regrets
                            ],
            'Sec To Compute': [bo_elapsed_seconds, tpe_elapsed_seconds, smac_elapsed_seconds, rs_elapsed_seconds]
        }

        all_patients_results.append(pd.DataFrame(comparison_data))
        all_convergence_results.append(
            pd.DataFrame(convergence_rows)
        )

        patient_and_machine_startSeed += 1

    return all_patients_results, all_convergence_results

**BASELINE FAMILY PATIENTS**

In [ ]:
baseline_family_test, baseline_convergence = run_optimization_comparison(
    num_simulation_trials= 40,
    amount_of_patients= 10,
    patient_and_machine_startSeed=10000,
    patient_type=baseline_family,
)



In [ ]:
from IPython.display import FileLink, display

if not isinstance(baseline_family_test, (list, tuple)) or not baseline_family_test:
    raise ValueError("baseline_family_test must be a non-empty list of per-profile DataFrames.")

baseline_family_results_df = pd.concat(
    [
        result.assign(
            **{
                "Profile Index": profile_index,
                "Profile ID": result["Surface Family"].astype(str)
                + "::"
                + result["Seed"].astype(str),
                "Iteration Budget": 40,
            }
        )
        for profile_index, result in enumerate(baseline_family_test, start=1)
    ],
    ignore_index=True,
)

baseline_family_convergence_df = pd.concat(
    [
        convergence.assign(**{"Profile Index": profile_index})
        for profile_index, convergence in enumerate(baseline_convergence, start=1)
    ],
    ignore_index=True,
)

baseline_family_results_path = "baseline_family_results.csv"
baseline_family_convergence_path = "baseline_family_convergence.csv"
baseline_family_results_df.to_csv(baseline_family_results_path, index=False)
baseline_family_convergence_df.to_csv(baseline_family_convergence_path, index=False)

print(
    f"Exported {len(baseline_family_results_df):,} profile-method rows "
    f"and {len(baseline_family_convergence_df):,} query rows."
)
display(
    FileLink(
        baseline_family_results_path,
        result_html_prefix="Download baseline family results: ",
    )
)
display(
    FileLink(
        baseline_family_convergence_path,
        result_html_prefix="Download baseline family convergence: ",
    )
)

In [ ]:
import itertools
import numpy as np
import pandas as pd
from scipy import stats

baseline_budget = 40
baseline_name = (
    baseline_family["name"]
    if "baseline_family" in globals()
    else "quick_rise_slow_fall"
)

if "baseline_family_test" not in globals():
    raise RuntimeError(
        "Run the baseline_family_test cell before running this baseline analysis."
    )

if "baseline_family_results_df" not in globals():
    baseline_family_results_df = pd.concat(
        [
            result.assign(
                **{
                    "Profile Index": profile_index,
                    "Profile ID": result["Surface Family"].astype(str)
                    + "::"
                    + result["Seed"].astype(str),
                    "Iteration Budget": baseline_budget,
                }
            )
            for profile_index, result in enumerate(
                baseline_family_test,
                start=1,
            )
        ],
        ignore_index=True,
    )

if "Iteration Budget" not in baseline_family_results_df.columns:
    baseline_family_results_df["Iteration Budget"] = baseline_budget

baseline_analysis_df = baseline_family_results_df[
    baseline_family_results_df["Iteration Budget"] == baseline_budget
].copy()

if set(baseline_analysis_df["Surface Family"].unique()) != {baseline_name}:
    raise ValueError(
        "The selected data contain families other than the baseline family."
    )

required_columns = {
    "Profile ID",
    "Method",
    "Best Possible Score",
    "Selected Score (Noise-Free)",
}
missing_columns = required_columns.difference(baseline_analysis_df.columns)
if missing_columns:
    raise ValueError(f"Missing baseline analysis columns: {sorted(missing_columns)}")

baseline_analysis_df["Normalized Regret"] = (
    baseline_analysis_df["Best Possible Score"]
    - baseline_analysis_df["Selected Score (Noise-Free)"]
) / baseline_analysis_df["Best Possible Score"]

method_order = [
    "Bayesian Optimization",
    "TPE Optimization",
    "SMAC Optimization",
    "Random Search",
]
methods = [
    method
    for method in method_order
    if method in baseline_analysis_df["Method"].unique()
]
if len(methods) != 4:
    raise ValueError(f"Expected all four methods; found {methods}.")

duplicate_keys = baseline_analysis_df.duplicated(
    subset=["Profile ID", "Method"],
    keep=False,
)
if duplicate_keys.any():
    raise ValueError(
        "Each baseline profile-method combination must have exactly one result."
    )

baseline_wide = baseline_analysis_df.pivot(
    index="Profile ID",
    columns="Method",
    values="Normalized Regret",
).reindex(columns=methods).dropna()

if len(baseline_wide) < 2:
    raise ValueError("At least two complete paired baseline profiles are required.")


def baseline_omnibus_permutation_test(
    wide,
    n_permutations=20000,
    seed=20260913,
):
    values = wide.to_numpy(dtype=float)
    observed_statistic = float(np.ptp(values.mean(axis=0)))
    rng = np.random.default_rng(seed)
    permuted_statistics = np.empty(n_permutations)

    for permutation_index in range(n_permutations):
        permuted_values = np.array(
            [rng.permutation(profile_values) for profile_values in values]
        )
        permuted_statistics[permutation_index] = np.ptp(
            permuted_values.mean(axis=0)
        )

    p_value = (
        1
        + np.count_nonzero(permuted_statistics >= observed_statistic)
    ) / (n_permutations + 1)
    return observed_statistic, p_value


def baseline_paired_sign_flip_test(
    first,
    second,
    n_permutations=20000,
    seed=20260913,
):
    differences = np.asarray(first, dtype=float) - np.asarray(second, dtype=float)
    observed_difference = float(np.mean(differences))
    rng = np.random.default_rng(seed)
    signs = rng.choice(
        np.array([-1.0, 1.0]),
        size=(n_permutations, len(differences)),
    )
    permuted_means = np.mean(signs * differences, axis=1)
    p_value = (
        1
        + np.count_nonzero(
            np.abs(permuted_means) >= abs(observed_difference)
        )
    ) / (n_permutations + 1)
    return observed_difference, p_value


def baseline_bootstrap_ci(
    differences,
    n_bootstrap=10000,
    seed=20260913,
):
    differences = np.asarray(differences, dtype=float)
    rng = np.random.default_rng(seed)
    indices = rng.integers(
        0,
        len(differences),
        size=(n_bootstrap, len(differences)),
    )
    bootstrap_means = differences[indices].mean(axis=1)
    return tuple(np.quantile(bootstrap_means, [0.025, 0.975]))


def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    sorted_p_values = p_values[order]
    adjusted_sorted = np.maximum.accumulate(
        (len(sorted_p_values) - np.arange(len(sorted_p_values)))
        * sorted_p_values
    )
    adjusted = np.empty_like(adjusted_sorted)
    adjusted[order] = np.minimum(adjusted_sorted, 1.0)
    return adjusted


friedman_statistic, friedman_p_value = stats.friedmanchisquare(
    *[baseline_wide[method].to_numpy() for method in methods]
)
omnibus_statistic, omnibus_p_value = baseline_omnibus_permutation_test(
    baseline_wide,
    seed=20260913 + baseline_budget,
)
kendalls_w = friedman_statistic / (
    len(baseline_wide) * (len(methods) - 1)
)

baseline_omnibus_df = pd.DataFrame(
    [
        {
            "Surface Family": baseline_name,
            "Iteration Budget": baseline_budget,
            "N Paired Profiles": len(baseline_wide),
            "Endpoint": "Noise-free normalized regret",
            "Friedman Statistic": friedman_statistic,
            "Friedman P Value": friedman_p_value,
            "Kendalls W": kendalls_w,
            "Permutation Range Statistic": omnibus_statistic,
            "Paired Permutation P Value": omnibus_p_value,
        }
    ]
)

pairwise_rows = []
for pair_index, (method_a, method_b) in enumerate(
    itertools.combinations(methods, 2)
):
    differences = (
        baseline_wide[method_a].to_numpy()
        - baseline_wide[method_b].to_numpy()
    )
    mean_difference, p_value = baseline_paired_sign_flip_test(
        baseline_wide[method_a],
        baseline_wide[method_b],
        seed=20300000 + pair_index,
    )
    ci_low, ci_high = baseline_bootstrap_ci(
        differences,
        seed=20400000 + pair_index,
    )
    difference_sd = np.std(differences, ddof=1)
    pairwise_rows.append(
        {
            "Surface Family": baseline_name,
            "Iteration Budget": baseline_budget,
            "Method A": method_a,
            "Method B": method_b,
            "N Paired Profiles": len(differences),
            "Mean Regret Difference (A - B)": mean_difference,
            "95% Bootstrap CI Low": ci_low,
            "95% Bootstrap CI High": ci_high,
            "Paired Effect Size dz": (
                mean_difference / difference_sd
                if difference_sd > 0
                else np.nan
            ),
            "P Method A Has Lower Regret": np.mean(differences < 0),
            "P Value": p_value,
        }
    )

baseline_pairwise_df = pd.DataFrame(pairwise_rows)
baseline_pairwise_df["Holm Adjusted P Value"] = holm_adjust(
    baseline_pairwise_df["P Value"]
)

baseline_summary_df = (
    baseline_analysis_df.groupby("Method")["Normalized Regret"]
    .agg(
        N_Profiles="count",
        Mean_Regret="mean",
        SD_Regret="std",
        Median_Regret="median",
        Q25_Regret=lambda values: values.quantile(0.25),
        Q75_Regret=lambda values: values.quantile(0.75),
    )
    .reindex(methods)
    .reset_index()
)

print(
    f"Baseline family: {baseline_name}; "
    f"{len(baseline_wide)} paired profiles; "
    f"{baseline_budget} queries."
)
print("Lower normalized regret is better.")
print("\n--- Baseline descriptive statistics ---")
display(baseline_summary_df)
print("\n--- Baseline omnibus tests ---")
display(baseline_omnibus_df)
print("\n--- Baseline paired post-hoc permutation tests ---")
display(baseline_pairwise_df)

In [ ]:
from IPython.display import FileLink, display

if "baseline_family_results_df" not in globals():
    raise RuntimeError(
        "Run the baseline export cell before running this download cell."
    )

if "baseline_family_convergence_df" not in globals():
    if "baseline_convergence" not in globals():
        raise RuntimeError(
            "baseline_convergence is required to create the convergence CSV."
        )
    baseline_family_convergence_df = pd.concat(
        [
            convergence.assign(**{"Profile Index": profile_index})
            for profile_index, convergence in enumerate(
                baseline_convergence,
                start=1,
            )
        ],
        ignore_index=True,
    )

baseline_download_files = {
    "Baseline profile results": (
        "baseline_family_results.csv",
        baseline_family_results_df,
    ),
    "Baseline convergence results": (
        "baseline_family_convergence.csv",
        baseline_family_convergence_df,
    ),
    "Baseline descriptive statistics": (
        "baseline_summary_statistics.csv",
        baseline_summary_df,
    ),
    "Baseline omnibus tests": (
        "baseline_omnibus_tests.csv",
        baseline_omnibus_df,
    ),
    "Baseline paired post-hoc tests": (
        "baseline_pairwise_tests.csv",
        baseline_pairwise_df,
    ),
}

for _, (path, frame) in baseline_download_files.items():
    frame.to_csv(path, index=False)

print("Baseline CSV files are ready for download:")
for label, (path, _) in baseline_download_files.items():
    display(FileLink(path, result_html_prefix=f"{label}: "))

In [ ]:
slow_rise_quick_fall_test, slow_rise_quick_fall_test_convergence = run_optimization_comparison(
    num_simulation_trials= 40,
    amount_of_patients= 40,
    patient_and_machine_startSeed=10000,
    patient_type= slow_rise_quick_fall_family,
)

In [ ]:
symmetric_test, symmetric_test_convergence= run_optimization_comparison(
    num_simulation_trials= 40,
    amount_of_patients= 40,
    patient_and_machine_startSeed=10000,
    patient_type= symmetric_family,
)

**THREE OPTIMA PATIENTS**

In [ ]:
three_optima_family_test, three_optima_convergence = run_optimization_comparison(
    num_simulation_trials= 40,
    amount_of_patients= 40,
    patient_and_machine_startSeed=10000,
    patient_type= three_optima_family,
)

**BROAD AND NARROW PEAKS PATIENTS**

In [ ]:
broad_peaks_family_test, broad_peaks_convergence = run_optimization_comparison(
    num_simulation_trials= 40,
    amount_of_patients= 40,
    patient_and_machine_startSeed=10000,
    patient_type= broad_peaks_family,
)
sharp_peaks_family_test, sharp_peaks_convergence = run_optimization_comparison(
    num_simulation_trials= 40,
    amount_of_patients= 40,
    patient_and_machine_startSeed=10000,
    patient_type= sharp_peaks_family,
)

**HETEROSCEDASTIC & HEAVY-TAILED NOISE & UNIQUE PARAMETER-DEPENDENCE PATIENTS**

In [ ]:
heteroscedastic_family_test, heteroscedastic_convergence = run_optimization_comparison(
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    patient_type=heteroscedastic_noise_family,
)

heavy_tailed_noise_family_test, heavy_tailed_noise_convergence = run_optimization_comparison(
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    patient_type=heavy_tailed_noise_family,
)

unique_parameter_interaction_family_test, unique_parameter_interaction_convergence = run_optimization_comparison(
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    patient_type=unique_parameter_interaction_family,
)

In [ ]:
final_comparison_df = pd.concat(
    baseline_family_test,
    three_optima_family_test,
    broad_peaks_family_test,
    sharp_peaks_family_test,
    heteroscedastic_family_test,
    heavy_tailed_noise_family_test,
    unique_parameter_interaction_family_test,
    slow_rise_quick_fall_test,
    ignore_index=True,
)

convergence_df = pd.concat(
    [
        *baseline_convergence,
        *three_optima_convergence,
        *broad_peaks_convergence,
        *sharp_peaks_convergence,
        *heteroscedastic_convergence,
        *heavy_tailed_noise_convergence,
        *unique_parameter_interaction_convergence,
    ],
    ignore_index=True,
)

print("\n--- Combined Optimization Comparison Results ---")
display(final_comparison_df.head())
final_comparison_df.to_csv('combined_optimization_results.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt

trajectory_summary = (
    convergence_df
    .groupby(
        [
            "Surface Family",
            "Method",
            "Iteration Budget",
            "Query",
        ]
    )["Current Incumbent Noise-Free Score"]
    .agg(
        Median="median",
        Q25=lambda values: values.quantile(0.25),
        Q75=lambda values: values.quantile(0.75),
    )
    .reset_index()
)

families = trajectory_summary["Surface Family"].unique()
n_columns = 2
n_rows = int(np.ceil(len(families) / n_columns))

figure, axes = plt.subplots(
    n_rows,
    n_columns,
    figsize=(14, 4 * n_rows),
    squeeze=False,
)
axes = axes.ravel()

for axis, family in zip(axes, families):
    family_data = trajectory_summary[
        trajectory_summary["Surface Family"] == family
    ]

    for method, method_data in family_data.groupby("Method"):
        method_data = method_data.sort_values("Query")
        queries = method_data["Query"].to_numpy()

        axis.plot(
            queries,
            method_data["Median"].to_numpy(),
            label=method,
        )
        axis.fill_between(
            queries,
            method_data["Q25"].to_numpy(),
            method_data["Q75"].to_numpy(),
            alpha=0.15,
        )

    axis.set_title(family)
    axis.set_xlabel("Query")
    axis.set_ylabel("Current Incumbent Noise-Free Score")
    axis.grid(alpha=0.25)

for axis in axes[len(families):]:
    axis.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
figure.legend(handles, labels, loc="lower center", ncol=2)
figure.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

convergence_df.to_csv(
    "optimization_convergence_results.csv",
    index=False,
)

## Statistics about Optimization Methods


In [ ]:
import itertools
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.power import TTestPower

analysis_df = final_comparison_df.copy()

# Use the post-optimization evaluation, not the largest noisy observation, for the primary analysis.
score_candidates = [
    "Selected Score (Noise-Free)",
    "Independent Reevaluation Score",
    "Best Computed Score",
]
score_column = next(
    (
        column
        for column in score_candidates
        if column in analysis_df.columns and not analysis_df[column].isna().all()
    ),
    None,
)
if score_column is None:
    raise ValueError(
        "The results need a selected-setting reevaluation score before statistical analysis."
    )
if score_column == "Best Computed Score":
    print(
        "WARNING: only Best Computed Score is available. These results are exploratory "
        "because this is the optimizer's best observed score, not an independent or "
        "noise-free reevaluation."
    )

# All methods were run on the same generated profile in each loop iteration.
if "Surface Family" not in analysis_df.columns or "Seed" not in analysis_df.columns:
    raise ValueError("Surface Family and Seed are required to identify paired profiles.")
analysis_df["Profile ID"] = (
    analysis_df["Surface Family"].astype(str)
    + "::"
    + analysis_df["Seed"].astype(str)
)

# The current notebook runs only the 40-query condition. Add an explicit column before
# combining results from 20- and 40-query runs.
if "Iteration Budget" not in analysis_df.columns:
    analysis_df["Iteration Budget"] = 40
    print("Iteration Budget was absent; assigned 40 for the current notebook run.")

analysis_df["Score Regret"] = (
    analysis_df["Best Possible Score"] - analysis_df[score_column]
)
analysis_df["Normalized Score Regret"] = (
    analysis_df["Score Regret"] / analysis_df["Best Possible Score"]
)

method_order = [
    "Bayesian Optimization",
    "TPE Optimization",
    "SMAC Optimization",
    "Random Search",
]
methods = [method for method in method_order if method in analysis_df["Method"].unique()]
if len(methods) != 4:
    raise ValueError(f"Expected all four methods; found {methods}.")

regret_tolerance = 0.05  # Prespecify and justify this threshold in the manuscript.


def paired_wide(frame, budget):
    subset = frame[frame["Iteration Budget"] == budget]
    duplicate_keys = subset.duplicated(
        subset=["Profile ID", "Method"], keep=False
    )
    if duplicate_keys.any():
        raise ValueError(
            "Each profile-method-budget combination must have exactly one result."
        )
    wide = subset.pivot(index="Profile ID", columns="Method", values=score_column)
    return wide.reindex(columns=methods).dropna()


def paired_sign_flip_test(first, second, n_permutations=20000, seed=20260913):
    differences = np.asarray(first, dtype=float) - np.asarray(second, dtype=float)
    observed_difference = float(np.mean(differences))
    rng = np.random.default_rng(seed)
    signs = rng.choice(
        np.array([-1.0, 1.0]),
        size=(n_permutations, len(differences)),
    )
    permuted_means = np.mean(signs * differences, axis=1)
    p_value = (
        1 + np.count_nonzero(np.abs(permuted_means) >= abs(observed_difference))
    ) / (n_permutations + 1)
    return observed_difference, p_value


def paired_bootstrap_ci(differences, n_bootstrap=10000, seed=20260913):
    differences = np.asarray(differences, dtype=float)
    rng = np.random.default_rng(seed)
    indices = rng.integers(
        0,
        len(differences),
        size=(n_bootstrap, len(differences)),
    )
    bootstrap_means = differences[indices].mean(axis=1)
    return tuple(np.quantile(bootstrap_means, [0.025, 0.975]))


def omnibus_permutation_test(wide, n_permutations=20000, seed=20260913):
    values = wide.to_numpy(dtype=float)
    observed_statistic = float(np.ptp(values.mean(axis=0)))
    rng = np.random.default_rng(seed)
    permuted_statistics = np.empty(n_permutations)
    for permutation_index in range(n_permutations):
        permuted_values = np.array(
            [rng.permutation(profile_values) for profile_values in values]
        )
        permuted_statistics[permutation_index] = np.ptp(
            permuted_values.mean(axis=0)
        )
    p_value = (
        1
        + np.count_nonzero(permuted_statistics >= observed_statistic)
    ) / (n_permutations + 1)
    return observed_statistic, p_value


summary_rows = []
omnibus_rows = []
pairwise_rows = []
power_rows = []

for budget in sorted(analysis_df["Iteration Budget"].unique()):
    wide = paired_wide(analysis_df, budget)
    if len(wide) < 2:
        continue

    omnibus_statistic, omnibus_p = omnibus_permutation_test(
        wide,
        seed=20260913 + int(budget),
    )
    friedman_statistic, friedman_p = stats.friedmanchisquare(
        *[wide[method].to_numpy() for method in methods]
    )
    omnibus_rows.append(
        {
            "Iteration Budget": budget,
            "N Paired Profiles": len(wide),
            "Permutation Range Statistic": omnibus_statistic,
            "Paired Permutation P Value": omnibus_p,
            "Friedman Statistic (Sensitivity)": friedman_statistic,
            "Friedman P Value (Sensitivity)": friedman_p,
        }
    )

    for method in methods:
        method_rows = analysis_df[
            (analysis_df["Iteration Budget"] == budget)
            & (analysis_df["Method"] == method)
        ]
        summary_rows.append(
            {
                "Iteration Budget": budget,
                "Method": method,
                "N Profiles": len(wide),
                "Mean Score": wide[method].mean(),
                "SD Score": wide[method].std(ddof=1),
                "Median Score": wide[method].median(),
                "IQR Score": wide[method].quantile(0.75)
                - wide[method].quantile(0.25),
                "Mean Regret": method_rows["Score Regret"].mean(),
                "Median Regret": method_rows["Score Regret"].median(),
                "P Regret <= Tolerance": (
                    method_rows["Normalized Score Regret"] <= regret_tolerance
                ).mean(),
            }
        )

    for pair_index, (first_method, second_method) in enumerate(
        itertools.combinations(methods, 2)
    ):
        differences = (
            wide[first_method].to_numpy() - wide[second_method].to_numpy()
        )
        mean_difference, p_value = paired_sign_flip_test(
            wide[first_method],
            wide[second_method],
            seed=20300000 + int(budget) * 100 + pair_index,
        )
        ci_low, ci_high = paired_bootstrap_ci(
            differences,
            seed=20400000 + int(budget) * 100 + pair_index,
        )
        difference_sd = np.std(differences, ddof=1)
        pairwise_rows.append(
            {
                "Iteration Budget": budget,
                "Method A": first_method,
                "Method B": second_method,
                "N Paired Profiles": len(differences),
                "Mean Score Difference (A - B)": mean_difference,
                "95% Bootstrap CI Low": ci_low,
                "95% Bootstrap CI High": ci_high,
                "Paired Effect Size dz": mean_difference / difference_sd
                if difference_sd > 0
                else np.nan,
                "P A Better Than B": np.mean(differences > 0),
                "P Value": p_value,
            }
        )

    for expected_dz in (0.2, 0.5, 0.8):
        power_rows.append(
            {
                "Iteration Budget": budget,
                "N Paired Profiles": len(wide),
                "Assumed Paired Effect dz": expected_dz,
                "Approximate Power (Holm-adjusted alpha)": TTestPower().power(
                    effect_size=expected_dz,
                    nobs=len(wide),
                    alpha=0.05 / 6,
                    alternative="two-sided",
                ),
            }
        )

summary_df = pd.DataFrame(summary_rows)
omnibus_df = pd.DataFrame(omnibus_rows)
pairwise_df = pd.DataFrame(pairwise_rows)
power_df = pd.DataFrame(power_rows)
if not pairwise_df.empty:
    pairwise_df["Holm Adjusted P Value"] = np.nan
    for budget in pairwise_df["Iteration Budget"].unique():
        mask = pairwise_df["Iteration Budget"] == budget
        pairwise_df.loc[mask, "Holm Adjusted P Value"] = multipletests(
            pairwise_df.loc[mask, "P Value"],
            method="holm",
        )[1]

print(f"Primary score column: {score_column}")
print("\n--- Paired descriptive statistics ---")
display(summary_df)
print("\n--- Paired omnibus tests ---")
display(omnibus_df)
print("\n--- Paired pairwise permutation tests with Holm correction ---")
display(pairwise_df)
print("\n--- Prospective power sensitivity (not post-hoc observed power) ---")
display(power_df)

if "convergence_df" not in globals():
    print(
        "\nConvergence trajectories are not available. Save one row per profile, "
        "method, iteration, and cumulative best noise-free reevaluation score "
        "during each optimization run before making the convergence plot."
    )